In [ ]:
Prince Kumar

# 📘 Mini Project: Predicting next word using LSTM/GRU

##  Objective
In this mini-project, we’ll build a simple **semantic keyword extractor** using the **SentenceTransformer** model.  
Given any input text, the model will find the **most relevant keywords** (topics) based on **cosine similarity** between the text and a predefined list of keywords.


In [ ]:
from datasets import load_dataset

df = load_dataset("wikitext", "wikitext-2-v1")
train = df["train"]["text"]
valid = df["validation"]["text"]
test = df["test"]["text"]


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense


#For preprocessing
import re
import pandas as pd
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
df.shape

{'test': (4358, 1), 'train': (36718, 1), 'validation': (3760, 1)}

In [ ]:
display(df['train'].to_pandas().head(10))

,text
0,
1,= Valkyria Chronicles III = \n
2,
3,Senjō no Valkyria 3 : <unk> Chronicles ( Japa...
4,"The game began development in 2010 , carrying..."
5,"It met with positive sales in Japan , and was..."
6,
7,= = Gameplay = = \n
8,
9,"As with previous <unk> Chronicles games , Val..."


In [ ]:
texts = df["train"]["text"]
texts


Column(['', ' = Valkyria Chronicles III = \n', '', ' Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . <unk> the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " <unk> Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more <unk> for 

In [ ]:
texts = texts


In [ ]:
def clean_wiki_lines(texts):
    cleaned = []
    for line in texts:
        line = line.strip()

        # remove empty lines
        if not line:
            continue

        # remove section headings
        if re.match(r"^=+.*=+$", line):
            continue

        # fix wiki artifacts
        line = line.replace("@-@", "-")

        cleaned.append(line)
    return cleaned


In [ ]:
texts = clean_wiki_lines(texts)


In [ ]:
sentences = []
for text in texts:
    sents = sent_tokenize(text)
    sentences.extend(sents)


In [ ]:
def normalize_text(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-zA-Z0-9.,!? ]+", "", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence


In [ ]:
sentences = [normalize_text(s) for s in sentences]


In [ ]:
tokenized_sentences = [word_tokenize(s) for s in sentences]


In [ ]:
MIN_LEN = 5
MAX_LEN = 30

tokenized_sentences = [
    s for s in tokenized_sentences
    if MIN_LEN <= len(s) <= MAX_LEN
]


In [ ]:
VOCAB_SIZE = 10000

word_freq = Counter()
for sent in tokenized_sentences:
    word_freq.update(sent)

most_common_words = word_freq.most_common(VOCAB_SIZE - 2)
word2idx = {word: idx+2 for idx, (word, _) in enumerate(most_common_words)}

# special tokens
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1

idx2word = {idx: word for word, idx in word2idx.items()}


In [ ]:
encoded_sentences = []
for sent in tokenized_sentences:
    encoded = [word2idx.get(word, word2idx["<UNK>"]) for word in sent]
    encoded_sentences.append(encoded)


In [ ]:
print("Total sentences:", len(encoded_sentences))
print("Vocabulary size:", len(word2idx))
print("Sample sentence (encoded):", encoded_sentences[0])
print("Sample sentence (decoded):", [idx2word[i] for i in encoded_sentences[0]])


Total sentences: 57663
Vocabulary size: 10000
Sample sentence (encoded): [1, 75, 4072, 78, 6, 4747, 745, 78, 4, 5473, 3]
Sample sentence (decoded): ['<UNK>', 'no', 'valkyria', '3', 'unk', 'chronicles', 'japanese', '3', ',', 'lit', '.']


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

input_sequences = []

for sentence in encoded_sentences:
    for i in range(1, len(sentence)):
        input_sequences.append(sentence[:i+1])


In [ ]:
print(max(len(seq) for seq in input_sequences))

30


In [ ]:
MAX_SEQ_LEN = max(len(seq) for seq in input_sequences)

padded_sequences = pad_sequences(
    input_sequences,
    maxlen=MAX_SEQ_LEN,
    padding="pre"
)


In [ ]:
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]


In [ ]:
y = y



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

EMBEDDING_DIM = 100

model = Sequential([
    Embedding(
        input_dim=len(word2idx),

        output_dim=EMBEDDING_DIM,
        input_length=X.shape[1]
    ),
    LSTM(64, dropout=0.2,)
,
  Dense(len(word2idx), activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


In [ ]:
history = model.fit(
    X, y,
    epochs=20,
    batch_size=64,
    validation_split=0.1
)


Epoch 1/20
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 117s 8ms/step - accuracy: 0.1368 - loss: 6.1572 - val_accuracy: 0.1929 - val_loss: 5.7007
Epoch 2/20
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 140s 8ms/step - accuracy: 0.1990 - loss: 5.3354 - val_accuracy: 0.2019 - val_loss: 5.5651
Epoch 3/20
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 114s 8ms/step - accuracy: 0.2125 - loss: 5.0765 - val_accuracy: 0.2065 - val_loss: 5.5074
Epoch 4/20
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 114s 8ms/step - accuracy: 0.2203 - loss: 4.9187 - val_accuracy: 0.2075 - val_loss: 5.4923
Epoch 5/20
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 114s 8ms/step - accuracy: 0.2267 - loss: 4.8150 - val_accuracy: 0.2091 - val_loss: 5.4894
Epoch 6/20
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 114s 8ms/step - accuracy: 0.2307 - loss: 4.7351 - val_accuracy: 0.2082 - val_loss: 5.4869
Epoch 7/20
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 114s 8ms/step - accuracy: 0.2343 - loss: 4.6783 - val_accuracy: 0.2085 - val_loss: 5.4912
Epoch 8/20
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 114s 8ms/step - ac

In [ ]:
history = model.fit( #model was overfitting so i stopped this training
    X, y,
    epochs=20,
    batch_size=64,
    validation_split=0.1
)


Epoch 1/30
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 126s 8ms/step - accuracy: 0.1416 - loss: 6.1242 - val_accuracy: 0.1983 - val_loss: 5.6281
Epoch 2/30
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 119s 8ms/step - accuracy: 0.2020 - loss: 5.2992 - val_accuracy: 0.2100 - val_loss: 5.4703
Epoch 3/30
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 120s 8ms/step - accuracy: 0.2187 - loss: 5.0109 - val_accuracy: 0.2142 - val_loss: 5.3999
Epoch 4/30
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 120s 8ms/step - accuracy: 0.2295 - loss: 4.8352 - val_accuracy: 0.2151 - val_loss: 5.3807
Epoch 5/30
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 121s 8ms/step - accuracy: 0.2373 - loss: 4.7033 - val_accuracy: 0.2154 - val_loss: 5.3717
Epoch 6/30
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 120s 8ms/step - accuracy: 0.2438 - loss: 4.6159 - val_accuracy: 0.2155 - val_loss: 5.3864
Epoch 7/30
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 121s 8ms/step - accuracy: 0.2484 - loss: 4.5370 - val_accuracy: 0.2147 - val_loss: 5.4070
Epoch 8/30
14660/14660 ━━━━━━━━━━━━━━━━━━━━ 120s 8ms/step - ac

KeyboardInterrupt: 

In [ ]:
model.save("next_word_model_best.h5")


In [ ]:
import pickle
with open("word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)
with open("idx2word.pkl", "wb") as f:
    pickle.dump(idx2word, f)
